# 06 - Entraînement du U-Net (Phase 3)

**Avant de lancer un entraînement complet, ce notebook permet de vérifier
cellule par cellule :**
1. Que CUDA (le GPU) est bien détecté
2. Que le modèle se construit correctement
3. Que les données se chargent correctement
4. Un essai rapide (peu d'epochs) avant l'entraînement complet

In [1]:
import torch, os, sys
print("Python utilise :", sys.executable)
print("Version torch :", torch.__version__)
print("Version CUDA compilee dans torch :", torch.version.cuda)
print("CUDA_VISIBLE_DEVICES :", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    print("Nombre de GPU detectes :", torch.cuda.device_count())
except Exception as e:
    print("Erreur lors de l'appel CUDA :", repr(e))

Python utilise : c:\Users\MSI\anaconda3\envs\myenv\python.exe
Version torch : 2.6.0+cu124
Version CUDA compilee dans torch : 12.4
CUDA_VISIBLE_DEVICES : None
Nombre de GPU detectes : 1


In [2]:
import sys
!{sys.executable} -m pip install torchvision --index-url https://download.pytorch.org/whl/cu124 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu124
   ---------------------------------------- 0.0/6.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.1 MB ? eta -:--:--
   -------- ------------------------------- 1.3/6.1 MB 3.4 MB/s eta 0:00:02
   ----------- ---------------------------- 1.8/6.1 MB 3.5 MB/s eta 0:00:02
   ----------------- ---------------------- 2.6/6.1 MB 3.5 MB/s eta 0:00:02
   ---------------------- ----------------- 3.4/6.1 MB 3.5 MB/s eta 0:00:01
   --------------------------- ------------ 4.2/6.1 MB 3.6 MB/s eta 0:00:01
   ------------------------------ --------- 4.7/6.1 MB 3.5 MB/s eta 0:00:01
   ----------------------------------- ---- 5.5/6.1 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 6.1/6.1 MB 3.5 MB/s eta 0:00:00


In [3]:
import sys
sys.path.append('../src/segmentation')

import json
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler

from dataset import PatchDataset
from model import build_model, count_parameters
from losses import DiceBCELoss

In [4]:
def build_optimizer(name, params, lr, weight_decay=0.0):
    """Construit l'optimiseur choisi (issu de la recherche du notebook 07)."""
    name = name.lower()
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Optimiseur inconnu : {name}")

## 0. Diagnostic GPU / CUDA

**La cellule la plus importante à vérifier avant tout.** Si `torch.cuda.is_available()`
affiche `False`, l'entraînement tournera sur CPU (très lent) — voir avec Claude
pour corriger l'installation de PyTorch avant de continuer.

In [5]:
print(f"Version de PyTorch : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")

# REQUIRE_GPU=True bloque volontairement l'execution si CUDA n'est pas
# detecte, au lieu de continuer silencieusement sur CPU (ce qui vient de
# faire perdre ~50 min sur un seul fold avant interruption manuelle).
# Mettre REQUIRE_GPU=False uniquement si tu acceptes deliberement un
# entrainement CPU (tres lent, deconseille pour ce projet).
REQUIRE_GPU = True

if torch.cuda.is_available():
    print(f"GPU detecte : {torch.cuda.get_device_name(0)}")
    print(f"Memoire GPU totale : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")
else:
    message = (
        "\nCUDA NON DETECTE - entrainement sur CPU environ 7 a 10x plus lent.\n"
        f"Version PyTorch installee : {torch.__version__}\n\n"
        "A verifier avant de continuer :\n"
        "  1. Dans PowerShell (hors Jupyter) : nvidia-smi\n"
        "     -> doit afficher la GTX 1650 Max-Q et une 'CUDA Version' >= 12.4\n"
        "     -> si la commande echoue : probleme de driver NVIDIA (reinstaller/mettre a jour)\n"
        "  2. Verifier que le bon environnement Conda est actif dans ce kernel :\n"
        "     import sys; print(sys.executable)\n"
        "     -> doit pointer vers ...\\anaconda3\\envs\\myenv\\...\n"
        "  3. Si le driver est trop ancien pour cu124, reinstaller torch avec une\n"
        "     version cu correspondant au driver, ex. :\n"
        "     pip install torch --index-url https://download.pytorch.org/whl/cu121 --force-reinstall\n"
    )
    print(message)
    if REQUIRE_GPU:
        raise RuntimeError(
            "Execution arretee : CUDA non disponible et REQUIRE_GPU=True. "
            "Corrige l'installation PyTorch/driver NVIDIA (voir message ci-dessus) "
            "avant de relancer, ou mets REQUIRE_GPU=False pour forcer un run CPU "
            "(deconseille)."
        )

Version de PyTorch : 2.6.0+cu124
CUDA disponible : True
GPU detecte : NVIDIA GeForce GTX 1650 with Max-Q Design
Memoire GPU totale : 4.3 Go


## 1. Configuration

In [6]:
PATCHES_DIR = '../data/processed/patches'
CHECKPOINTS_DIR = '../models/saved'
RESULTS_DIR = '../data/results'

# --- Hyperparametres issus de la recherche du notebook 07
# (recherche sur le fold Tracee1, 10 epochs, best_valid_loss=0.1812) ---
BATCH_SIZE = 8            # trouve meilleur que 8 ou 16 lors de la recherche
NUM_EPOCHS = 25           # nombre max d'epochs (l'early stopping arretera avant si besoin)
LEARNING_RATE = 3e-4      # 3x plus eleve que la valeur par defaut precedente (1e-4)
OPTIMIZER_NAME = "rmsprop"  # remplace Adam, trouve meilleur lors de la recherche
DICE_WEIGHT = 0.7         # plus de poids au Dice qu'a la BCE (defaut precedent : 0.5/0.5)
BCE_WEIGHT = 0.3
NUM_WORKERS = 0           # mettre 0 sous Windows si probleme de multiprocessing
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# WEIGHT_DECAY : la recherche rapide (10 epochs, 1 fold) trouvait 0.0
# "meilleur", mais cette recherche est trop courte pour capturer le
# surapprentissage qui apparait typiquement apres 15-20 epochs sur
# l'entrainement complet (deja observe sur les runs precedents : train_loss
# continue de baisser, valid_loss stagne/remonte). Priorite donnee ici a la
# generalisation plutot qu'au resultat brut de la recherche courte : on fixe
# volontairement une regularisation L2 non nulle.
WEIGHT_DECAY = 1e-4

# Early stopping : arrete l'entrainement si valid_loss ne s'ameliore plus
# depuis PATIENCE epochs consecutives. D'apres les resultats precedents,
# valid_loss plafonne generalement vers l'epoch 15-22 puis stagne/oscille
# (train_loss continue de baisser -> signe de surapprentissage). PATIENCE=7
# laisse une marge raisonnable pour ne pas s'arreter sur une simple
# oscillation ponctuelle, tout en coupant avant un surapprentissage marque.
PATIENCE = 10

# AMP (precision mixte fp16) DESACTIVEE : avec un encodeur pre-entraine +
# un decodeur initialise aleatoirement, les activations du decodeur peuvent
# deborder la plage fp16 (~65504) des le premier passage avant, meme avant
# tout entrainement -> inf -> nan sur la perte. Le fp32 a une plage bien plus
# large (~3e38) et evite ce probleme. A reactiver plus tard uniquement si
# besoin de vitesse/memoire, et seulement une fois l'entrainement stable.
USE_AMP = False

TRACEES = ["Tracee1", "Tracee2", "Tracee3", "Tracee4"]

print(f"Device utilise : {DEVICE}")
print(f"AMP (precision mixte) : {USE_AMP}")
print(f"Optimiseur : {OPTIMIZER_NAME} | LR : {LEARNING_RATE:.0e} | weight_decay : {WEIGHT_DECAY:.0e}")
print(f"Dice/BCE : {DICE_WEIGHT}/{BCE_WEIGHT} | batch_size : {BATCH_SIZE}")
print(f"Epochs max par fold : {NUM_EPOCHS}")
print(f"Patience early stopping : {PATIENCE}")

Device utilise : cuda
AMP (precision mixte) : False
Optimiseur : rmsprop | LR : 3e-04 | weight_decay : 1e-04
Dice/BCE : 0.7/0.3 | batch_size : 8
Epochs max par fold : 25
Patience early stopping : 10


## 2. Vérification du modèle (construction + formes)

In [7]:
model_test = build_model(pretrained=True).to(DEVICE)
total, trainable = count_parameters(model_test)
print(f"Modele cree : {total:,} parametres ({trainable:,} entrainables)")

dummy = torch.randn(2, 3, 256, 256).to(DEVICE)
output = model_test(dummy)
print(f"Entree {tuple(dummy.shape)} -> Sortie {tuple(output.shape)}")
assert output.shape == (2, 2, 256, 256)
print("OK : le modele fonctionne sur", DEVICE)

del model_test, dummy, output
if DEVICE == "cuda":
    torch.cuda.empty_cache()

Modele cree : 14,328,354 parametres (14,328,354 entrainables)
Entree (2, 3, 256, 256) -> Sortie (2, 2, 256, 256)
OK : le modele fonctionne sur cuda


## 3. Vérification du chargement des données

In [8]:
ds_test = PatchDataset(PATCHES_DIR)
print(f"{len(ds_test)} patchs trouves au total dans {PATCHES_DIR}")

img, mask = ds_test[0]
print(f"Image: {tuple(img.shape)} | Masque: {tuple(mask.shape)}")

for tracee in TRACEES:
    ds_excl = PatchDataset(PATCHES_DIR, include_tracees=[tracee])
    print(f"  {tracee}: {len(ds_excl)} patchs")

660 patchs trouves au total dans ../data/processed/patches
Image: (3, 256, 256) | Masque: (2, 256, 256)
  Tracee1: 140 patchs
  Tracee2: 200 patchs
  Tracee3: 132 patchs
  Tracee4: 188 patchs


## 4. Fonction d'entraînement d'un fold

Un "fold" = un entraînement complet avec un tracé mis de côté pour la
validation (jamais vu pendant l'entraînement).

In [9]:
def train_one_fold(fold_tracee_valid, epochs=NUM_EPOCHS, patience=PATIENCE):
    print(f"\n{'='*70}")
    print(f"FOLD - validation sur {fold_tracee_valid} (entrainement sur les 3 autres)")
    print(f"{'='*70}")

    train_ds = PatchDataset(PATCHES_DIR, exclude_tracees=[fold_tracee_valid])
    valid_ds = PatchDataset(PATCHES_DIR, include_tracees=[fold_tracee_valid])
    print(f"Entrainement: {len(train_ds)} patchs | Validation: {len(valid_ds)} patchs")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = build_model(pretrained=True).to(DEVICE)
    optimizer = build_optimizer(OPTIMIZER_NAME, model.parameters(), LEARNING_RATE, WEIGHT_DECAY)
    loss_fn = DiceBCELoss(dice_weight=DICE_WEIGHT, bce_weight=BCE_WEIGHT)
    scaler = GradScaler(enabled=(DEVICE == "cuda" and USE_AMP))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )  # reduit le LR de moitie si valid_loss stagne 3 epochs (complementaire de l'early stopping)

    history = {"train_loss": [], "valid_loss": []}
    best_valid_loss = float("inf")
    epochs_sans_amelioration = 0
    best_epoch = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        model.train()
        train_loss_total = 0.0
        for images, masks in train_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with autocast(device_type=DEVICE, enabled=(DEVICE == "cuda" and USE_AMP)):
                logits = model(images)
                loss = loss_fn(logits, masks)
            if not torch.isfinite(loss):
                print("  [ATTENTION] perte non-finie detectee sur un batch, ignore.")
                optimizer.zero_grad()
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss_total += loss.item() * images.size(0)
        train_loss = train_loss_total / len(train_ds)

        model.eval()
        valid_loss_total = 0.0
        with torch.no_grad():
            for images, masks in valid_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                with autocast(device_type=DEVICE, enabled=(DEVICE == "cuda" and USE_AMP)):
                    logits = model(images)
                    loss = loss_fn(logits, masks)
                valid_loss_total += loss.item() * images.size(0)
        valid_loss = valid_loss_total / len(valid_ds)
        scheduler.step(valid_loss)

        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid_loss)

        elapsed = time.time() - t0

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_epoch = epoch
            epochs_sans_amelioration = 0
            Path(CHECKPOINTS_DIR).mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), Path(CHECKPOINTS_DIR) / f"unet_fold_{fold_tracee_valid}_best.pt")
            marqueur = " *"  # nouveau meilleur modele sauvegarde
        else:
            epochs_sans_amelioration += 1
            marqueur = ""

        lr_actuel = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d}/{epochs} | train_loss={train_loss:.4f} | "
              f"valid_loss={valid_loss:.4f} | lr={lr_actuel:.1e} | {elapsed:.1f}s{marqueur}")

        if epochs_sans_amelioration >= patience:
            print(f"  [EARLY STOPPING] valid_loss n'a plus progresse depuis {patience} epochs "
                  f"(meilleure epoch : {best_epoch}, best_valid_loss={best_valid_loss:.4f}). Arret.")
            break

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return {
        "fold_valid_tracee": fold_tracee_valid,
        "best_valid_loss": best_valid_loss,
        "best_epoch": best_epoch,
        "stopped_epoch": epoch,
        "history": history,
    }

## 5. Essai sur UN seul fold


In [10]:
resultat_test = train_one_fold(fold_tracee_valid='Tracee1', epochs=NUM_EPOCHS)
print(f"\nEssai reussi. Meilleure perte de validation : {resultat_test['best_valid_loss']:.4f}")


FOLD - validation sur Tracee1 (entrainement sur les 3 autres)
Entrainement: 520 patchs | Validation: 140 patchs
Epoch   1/25 | train_loss=0.3862 | valid_loss=0.3941 | lr=3.0e-04 | 15.0s *
Epoch   2/25 | train_loss=0.2511 | valid_loss=0.3172 | lr=3.0e-04 | 13.6s *
Epoch   3/25 | train_loss=0.1996 | valid_loss=0.2724 | lr=3.0e-04 | 13.5s *
Epoch   4/25 | train_loss=0.1736 | valid_loss=0.2436 | lr=3.0e-04 | 13.7s *
Epoch   5/25 | train_loss=0.1628 | valid_loss=0.2467 | lr=3.0e-04 | 13.5s
Epoch   6/25 | train_loss=0.1471 | valid_loss=0.2203 | lr=3.0e-04 | 13.6s *
Epoch   7/25 | train_loss=0.1424 | valid_loss=0.2278 | lr=3.0e-04 | 14.0s
Epoch   8/25 | train_loss=0.1347 | valid_loss=0.2073 | lr=3.0e-04 | 13.9s *
Epoch   9/25 | train_loss=0.1310 | valid_loss=0.2035 | lr=3.0e-04 | 13.8s *
Epoch  10/25 | train_loss=0.1248 | valid_loss=0.2232 | lr=3.0e-04 | 13.9s
Epoch  11/25 | train_loss=0.1246 | valid_loss=0.2292 | lr=3.0e-04 | 14.2s
Epoch  12/25 | train_loss=0.1227 | valid_loss=0.2164 | lr=3

## 6. Entraînement complet — les 4 folds (leave-one-tracé-out)

In [11]:
all_results = [resultat_test]  # on reutilise le fold Tracee1 deja fait ci-dessus

for tracee in TRACEES[1:]:
    result = train_one_fold(fold_tracee_valid=tracee, epochs=NUM_EPOCHS)
    all_results.append(result)

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(RESULTS_DIR) / 'training_leave_one_out_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n{'='*70}")
print("RESUME DE LA VALIDATION LEAVE-ONE-TRACE-OUT")
print(f"{'='*70}")
for r in all_results:
    print(f"  Validation sur {r['fold_valid_tracee']:10s} -> meilleure perte de validation : "
          f"{r['best_valid_loss']:.4f} (epoch {r['best_epoch']}, arret a l'epoch {r['stopped_epoch']})")



FOLD - validation sur Tracee2 (entrainement sur les 3 autres)
Entrainement: 460 patchs | Validation: 200 patchs
Epoch   1/25 | train_loss=0.4137 | valid_loss=0.4611 | lr=3.0e-04 | 13.0s *
Epoch   2/25 | train_loss=0.2465 | valid_loss=0.2744 | lr=3.0e-04 | 12.9s *
Epoch   3/25 | train_loss=0.1956 | valid_loss=0.2168 | lr=3.0e-04 | 12.9s *
Epoch   4/25 | train_loss=0.1701 | valid_loss=0.2617 | lr=3.0e-04 | 13.2s
Epoch   5/25 | train_loss=0.1547 | valid_loss=0.1952 | lr=3.0e-04 | 13.1s *
Epoch   6/25 | train_loss=0.1465 | valid_loss=0.2566 | lr=3.0e-04 | 12.9s
Epoch   7/25 | train_loss=0.1357 | valid_loss=0.2380 | lr=3.0e-04 | 12.9s
Epoch   8/25 | train_loss=0.1321 | valid_loss=0.4229 | lr=3.0e-04 | 12.9s
Epoch   9/25 | train_loss=0.1336 | valid_loss=0.2140 | lr=1.5e-04 | 13.1s
Epoch  10/25 | train_loss=0.1174 | valid_loss=0.1936 | lr=1.5e-04 | 13.2s *
Epoch  11/25 | train_loss=0.1105 | valid_loss=0.1933 | lr=1.5e-04 | 12.9s *
Epoch  12/25 | train_loss=0.1123 | valid_loss=0.2074 | lr=1.5

## Prochaines étapes

1. Une fois l'entraînement complet terminé (NUM_EPOCHS relevé), analyser les
   courbes train_loss/valid_loss par fold pour détecter du surapprentissage
2. Extraction des points de contour depuis les masques prédits (Phase 5)
3. Ré-entraîner un modèle final sur les 4 tracés (plus de données) pour la
   version de production
